# Intro to AI Agents

---

In this notebook, we will introduce the concept of AI agents and how they can be used to solve problems. Unlike static LLM workflows, AI agents are dynamic and can adapt to a much wider range of problems. They are particularly well suited when we don't know the exact parameters of each user interaction. Some cases which may call for AI agents include:
- Goal oriented tasks such as researching a topic, analyzing data across many sources, or helping find the best product for a user.
- Tasks that may involve a wide variety of user inputs, such as document processing where there may be a wide variety of document formats and data that needs to be extracted.
- Tasks that may involve usage of various external tools or APIs, without a fixed sequential order of operations.

There are numerous frameworks available for implementing agents including [Amazon Bedrock](https://aws.amazon.com/bedrock/agents/), [CrewAI](https://www.crewai.com/), and [LangGraph](https://www.langchain.com/langgraph). The frameworks vary in their capabilities and complexity, but all provide a way to define and deploy agents that can interact with users and other systems.For this notebook, we will use a lightweight framework from Hugging Face called [smolagents](https://github.com/huggingface/smolagents). This framework provides a very simple and lightweight way to define agents that can interact with users and other systems, and is thus well suitable for quick prototyping and experimentation.


---

In [1]:
import sys
import os
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils.environment_validation import validate_environment, validate_model_access
validate_environment()

Validating base environment
Base environment validated successfully


Validating lab environment from requirements.txt ✨

ENVIRONMENT STATUS
✅  langchain==0.3.7 is installed
✅  langchain-aws==0.2.6 is installed
✅  langchain-community==0.3.5 is installed
✅  langchain-core==0.3.15 is installed
✅  langchain-text-splitters==0.3.2 is installed
✅  sqlalchemy is installed
✅  pypdf>=3.8,<4 is installed
✅  datasets is installed
✅  matplotlib is installed
✅  pymupdf  is installed
✅  xmltodict==0.13.0 is installed
✅  duckduckgo-search is installed
✅  yfinance>=0.2.54 is installed
✅  pandas-datareader is installed
✅  pysqlite3 is installed
✅  smolagents==1.9.2 is installed
✅  litellm==1.61.15 is installed

All required libraries are installed.🎉
You may proceed with the lab! 🚀

In [2]:
required_models = [
    "amazon.titan-embed-text-v1",
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    "us.amazon.nova-pro-v1:0",
]
validate_model_access(required_models)

MODEL ACCESS STATUS
✅  amazon.titan-embed-text-v1 is accessible
✅  us.anthropic.claude-3-5-haiku-20241022-v1:0 is accessible
✅  us.anthropic.claude-3-5-sonnet-20241022-v2:0 is accessible
✅  us.amazon.nova-pro-v1:0 is accessible

All required models are accessible.🎉
You may proceed with the lab! 🚀

## Simple Search Agent

Let's build a simple search agent that can search for information on the web. The agent will take a query from the user and return the top search results. We will use DuckDuckGo as the search engine for this agent.

We will be using the opensource [`smolagents`](https://github.com/huggingface/smolagents) library, which offers two types of agents:
- [CodeAgent](https://huggingface.co/papers/2402.01030): Invokes tools via generated python code snippets. This provides flexibility in terms of how the agent interacts with external tools as it's able to expand on the capabilities of the tools by incorporating custom code.
- ToolCallingAgent: Invokes tools by generating a JSON output that contains the tool name and parameters and the invocation parameters. This approach is more rigid as the model is limited to the capabilities of the tools it calls however it may prove to be more efficient and secure in some cases as it does not involve running arbitrary code.

As as an example if we have a tool that can reterieve stock data for a given stock symbol. If the agent receives a task that involves retrieving stock data for say "AAPL", "MSFT" and "GOOGL", the CodeAgent will write a code snippet with a simple for loop that will retrieve the stock data for each symbol. While the ToolCallingAgent will generate a JSON output that contains the tool name and parameters and the invocation parameters for each stock symbol. Additionally, if the tasks requires invoking additional tools to analyze the stock data, the CodeAgent can potentially tackle this in a single code snippet, while the ToolCallingAgent will need to make multiple calls to the tools which could be less efficient.

In [3]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, GradioUI, tool
from typing import List
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import statsmodels.api as sm
import numpy as np


In [12]:
MODEL_ID = "bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0"
model = LiteLLMModel(model_id=MODEL_ID, temperature=0)  # LiteLLMModel is smolagent's model agnostic API that instantiates a LLM client

In [5]:
# define as simple agent that has access to online
# https://huggingface.co/docs/smolagents/reference/agents#smolagents.CodeAgent

agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model)
agent.run("How many years would it take an average person to watch all of the content on Amazon prime video?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many years would it take an average person to watch all of the content on Amazon prime video?               │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0 ────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  web_search(query="How many hours of content are on Amazon Prime Video total")                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: ## Search Results

[Amazon Prime Video Statistics By Users, Revenue and Facts - Sci-Tech 
Today](https://www.sci-tech-today.com/stats/amazon-prime-video-statistics/)
As stated in Amazon Prime Video Statistics, the total number of Prime video viewers in Canada in 2019 was only 7 
million, and it is estimated to reach 14.7 million by the end of 2025.

[Amazon Prime Video Revenue and Usage Statistics 
(2025)](https://www.businessofapps.com/data/amazon-prime-video-statistics/)
Prime Video launched all the way back in 2006, originally called Amazon Unbox. At launch, it offered a way to store
TV series and movies purchased from Amazon, with an instant video subscription added in 2011 with 5,000 movies and 
TV shows available to watch. Unlike Netflix and other video streaming services, Prime Video was offered as part of 
the Amazon's Prime subscription. The goal was to ...

[Amazon Prime Video - statistics & facts | Statista](https://www.statista.com/topics/4740/amazon-prime-video/)
Premium Statistic Most enjoyed content on Amazon Prime Video in the U.S. 2017-2024 Premium Statistic Distribution 
of content on Amazon Prime Video worldwide 2024, by genre

[Amazon Prime Video Statistics 2023 - Users, Revenue and 
Facts](https://www.enterpriseappstoday.com/stats/amazon-prime-video-statistics.html)
57% of the people subscribed to Prime as it has video content. Amazon has published around 75 million videos on ...
a user watches prime video for 5 hours and 22 minutes. 39% of the adults in the United States of America are Amazon
Prime video members. There are a total of around 54.3% of users access primevideo.com via desktop and 45.7% of ...

[Media Streaming Stats You Should Know - Forbes 
Home](https://www.forbes.com/home-improvement/internet/streaming-stats/)
An overwhelming 99% of U.S. households now subscribe to at least one or more streaming services, with Netflix, 
Amazon Prime Video and Apple TV+ topping the charts.

[Amazon Prime Statistics 2025 - Subscribers & Revenue - Evoca](https://evoca.tv/amazon-prime-statistics/)
7 in 10 Prime members stream video on Amazon Prime Video every month; Amazon Prime Video is available in over 200 
countries. According to a 2022 survey, 15% of the respondents watched the content daily on Amazon Prime Video, 
while 46% stated that they had not used Amazon Prime Video for streaming content. Amazon Prime Video membership 
costs $8. ...

[Amount of original content on Amazon Prime Video 2019 | 
Statista](https://www.statista.com/statistics/883472/amazon-prime-video-original-content-hours/)
Number of hours of original content produced by Amazon Prime Video worldwide from 2012 to 2019 (in hours) [Graph], 
Advanced Television, March 17, 2020. [Online].

[Amazon Prime Video Statistics By Revenue, Users and 
Facts](https://www.coolest-gadgets.com/amazon-prime-video-statistics/)
Introduction. Amazon Prime Video Statistics: Amazon Prime Video is a streaming service from Amazon that includes a 
huge collection of movies, TV shows, and original content. Started in 2006, Prime Video is now available in more 
than 240 countries and regions. It offers many types of shows and movies, such as drama, comedy, action, and 
documentaries, to suit all tastes.

[Hours of content on UK SVOD platforms 2024 | 
Statista](https://www.statista.com/statistics/963341/hours-of-content-on-netflix-and-amazon-prime-video-united-king
dom-uk/)
In the United Kingdom, Amazon Prime Video recorded the highest number of content hours available in its library 
among selected SVOD services. As of May 2024, the amount was over 42.6 thousand hours.

[Amazon Prime Video Statistics and Facts - 
Market.us](https://market.us/statistics/online-video-and-streaming-sites/amazon-prime-video/)
In 2018, Amazon.com, Inc. was to spend over US$ 5 billion for Amazon content on Prime Video; In 2018, Netflix, Inc.
and Amazon.com, Inc. spent around US$ 11 billion on content, and it is expected to increase to around US$ 20 
billion per year split by 2023; I

[Step 0: Duration 7.47 seconds| Input tokens: 2,373 | Output tokens: 84]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Estimate based on previous calculation                                                                         
  total_content_hours = 213000  # From previous calculation                                                        
                                                                                                                   
  # Average daily viewing time (from previous search)                                                              
  avg_daily_viewing_hours = 3.15                                                                                   
                                                                                                                   
  # Calculate years to watch                                                                                       
  years_to_watch = total_content_hours / (avg_daily_viewing_hours * 365)                                           
                                                                                                                   
  # Round to two decimal places                                                                                    
  years_to_watch_rounded = round(years_to_watch, 2)                                                                
                                                                                                                   
  print(f"Estimated years to watch all Amazon Prime Video content: {years_to_watch_rounded} years")                
                                                                                                                   
  # Final answer                                                                                                   
  final_answer(years_to_watch_rounded)                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Estimated years to watch all Amazon Prime Video content: 185.26 years

Out - Final answer: 185.26

[Step 4: Duration 9.51 seconds| Input tokens: 22,856 | Output tokens: 981]

185.26

## Stock Analysis Agent
Let's look at a more complex example. We will build an agent that can analyze stock data. In addition to web search, this agent will have access to the following tools:
- `get_ticker_data`: This tool will retrieve stock data for a given stock symbol.
- `get_fred_data`: This tool will retrieve economic data from the Federal Reserve Economic Data (FRED) API.
- `run_ols_regression`: This tool will run an ordinary least squares regression which can be used to analyze the relationship between two variables

Defining tools is easy as we merely need to decorate a function with the `@tool` decorator. The function should contain a docstring in [Google-style](https://sphinxcontrib-napoleon.readthedocs.io/en/latest/example_google.html) format that describes the tool's inputs and outputs. Providing clear documentation is important as it informs the agent about the tool's capabilities and how to use it.

In [6]:
@tool
def get_ticker_data(
    tickers: List[str],
    start_date: str,
    end_date: str,
    metric: str = "all",
    sampling: str = "monthly",
) -> dict:
    """Downloads historical stock data from Yahoo Finance and returns it as a dictionary.

    Examples:
        >>> get_ticker_data(["AAPL"], "2023-01-01", "2023-12-31", "Close", "weekly")
        {"AAPL": [{"Date": "2023-01-06", "Close": 129.619995}, {"Date": "2023-01-13", "Close": 134.759995}, ...]}

        >>> get_ticker_data(["AAPL", "MSFT"], "2023-01-01", "2023-12-31", "all", "monthly")
        {"AAPL": [{"Date": "2023-01-31", "Open": 144.479996, "High": 147.229996, "Low": 141.320007, "Close": 144.289993, "Adj Close": 143.839996, "Volume": 77663600}, ...],
          "MSFT": [{"Date": "2023-01-31", "Open": 250.089996, "High": 256.25, "Low": 242.529999, "Close": 252.509995, "Adj Close": 251.873795, "Volume": 47146900}, ...]}

    Args:
        tickers: A list of stock ticker symbols (e.g., ["AAPL", "MSFT"]).
        start_date: The start date for the data (e.g., "2023-01-01").
        end_date: The end date for the data in YYYY-MM-DD format (e.g., "2023-12-31").
        metric:  If "all", returns all available data columns (Open, High, Low, Close, Volume).
            Otherwise, specifies a single metric to return (e.g., "Close"). Defaults to "all".
        sampling: The frequency of the data. Can be "daily", "weekly", or "monthly". Defaults to "monthly".

    Returns:
        dict: A dictionary where keys are ticker symbols and values are lists of historical data records.
             Each record is a dictionary containing 'Date' and the requested metrics.

    Raises:
        ValueError: If an invalid sampling frequency is provided.


    """

    df = yf.download(tickers, start=start_date, end=end_date)

    if metric != "all":
        df = df[metric]

    if sampling == "weekly":
        df = df.resample("W-SAT").last()
    elif sampling == "monthly":
        df = df.resample("ME").last()
    elif sampling == "quarterly":
        df = df.resample("QE").last()
    elif sampling == "daily":
        pass
    else:
        raise ValueError(
            "Invalid sampling frequency. Use 'daily', 'weekly', 'monthly', 'quarterly."
        )

    result = {}
    for ticker in tickers:
        if metric == "all":
            df_tick = df.loc[:, (slice(None), ticker)]
            df_tick.columns = df_tick.columns.droplevel("Ticker")
        else:
            df_tick = df.loc[:, ticker]
            df_tick = df_tick.to_frame(name=metric)
        df_tick = df_tick.reset_index()
        df_tick["Date"] = df_tick["Date"].dt.strftime("%Y-%m-%d")
        result[ticker] = df_tick.to_dict(orient="records")

    return result


@tool
def get_fred_data(
    series: str, start_date: str, end_date: str, sampling: str = "monthly"
) -> list[dict]:
    """Downloads data from the Federal Reserve Economic Data (FRED) database and returns it as dictionary.

    Examples:
        >>> get_fred_data("GDP", "2023-01-01", "2023-01-10")
        [{"Date": "2023-01-01", "GDP": 21.0}, {"Date": "2023-01-02", "GDP": 22.0}, ...]

    Args:
        series: The FRED series ID (e.g., "GDP").
        start_date: The start date for the data (e.g., "2023-01-01").
        end_date: The end date for the data in YYYY-MM-DD format (e.g., "2023-12-31").
        sampling: The frequency of the data. Can be "monthly", "quarterly", or "yearly". Defaults to "monthly".

    Returns:
        list: A list representing a list of dictionaries, where each dictionary contains 'Date' and the value of the FRED series.

    Raises:
        ValueError: If an invalid sampling frequency is provided.


    """
    df = pdr.data.DataReader(series, start=start_date, end=end_date, data_source="fred")

    if sampling == "monthly":
        df = df.resample("ME").last()
    elif sampling == "quarterly":
        df = df.resample("QE").last()
    elif sampling == "yearly":
        df = df.resample("YE").last()
    else:
        raise ValueError(
            "Invalid sampling frequency. Use 'monthly', 'quarterly', or 'yearly'."
        )

    df.reset_index(inplace=True)
    df["DATE"] = df["DATE"].dt.strftime("%Y-%m-%d")
    df.rename(columns={"DATE": "Date"}, inplace=True)

    result = df.to_dict(orient="records")

    return result


@tool
def run_ols_regression(y: List[float], X: List[float]) -> dict:
    """Runs a simple Ordinary Least Squares (OLS) regression.
    I using to compute beta, make sure the dates are aligned.

    Examples:
        >>> y = [1, 2, 3, 4, 5]
        >>> X = [2, 4, 5, 4, 5]
        >>> run_ols_regression(y, X)
        {"const": -0.4, "coef": 0.9}

    Args:
        y: The dependent variable.
        X: The independent variable(s).

    Returns:
        dict: A dictionary containing the constant and coefficient of the OLS regression.


    """
    X = np.array(X)
    y = np.array(y)
    X = sm.add_constant(X)
    model = sm.OLS(y, X)
    results = model.fit()
    params = results.params
    const, coef = params
    return {"const": const, "coef": coef}

In [7]:
# define the stock analysis agent
stock_analysis_agent = CodeAgent(
    tools=[get_ticker_data, get_fred_data, run_ols_regression, DuckDuckGoSearchTool()],
    model=model,
    name="stock_analyst_agent",
    description="A research agent that specializes in analyzing stock performance, computing technical indicators, and forecasting volatility.",
)

In [8]:
stock_analysis_agent.run("What immediate impact did Amazon's announcment of Alexa+ have on its stock price?")

╭───────────────────────────────────────── New run - stock_analyst_agent ─────────────────────────────────────────╮
│                                                                                                                 │
│ What immediate impact did Amazon's announcment of Alexa+ have on its stock price?                               │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0 ────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  web_search_result = web_search(query="Amazon Alexa+ announcement date stock market impact")                      
  print(web_search_result)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Amazon Stock Gains On Alexa+ Plans. Here's What's New With The Gen AI 
...](https://www.investors.com/news/technology/amazon-stock-alexa-update-generative-ai/)
The rollout for Alexa+ will start with the U.S. in the "next few weeks," according to the company announcement. On 
the stock market today , Amazon stock gained a half-percent to close at 214.25 ...

[Is Amazon Stock a Buy, Sell, or Hold Ahead of Alexa+ Launch? - 
MSN](https://www.msn.com/en-us/news/technology/is-amazon-stock-a-buy-sell-or-hold-ahead-of-alexa-launch/ar-AA1AaMmw
)
Amazon's latest earnings breakdown also highlights strong momentum across its core business segments. The North 
American division led the charge with a solid 10% year-over-year growth ...

[All-new Alexa+ and more: All the news from Amazon's 2025 devices 
event](https://www.aboutamazon.com/news/devices/amazon-2025-devices-alexa-event-live-updates)
Alexa+ costs $19.99 per month, but is free for all Prime members. Alexa+ will start rolling out in the U.S. in the 
next few weeks, and subsequently in waves in the coming months starting with households with Echo Show 8, 10, 15, 
and 21.

[Is Amazon Stock a Buy, Sell, or Hold Ahead of Alexa+ Launch? - 
Barchart.com](https://www.barchart.com/story/news/31215644/is-amazon-stock-a-buy-sell-or-hold-ahead-of-alexa-launch
)
Alexa's AI makeover marks a bold new chapter for Amazon, but will it be a game-changer for the stock or just 
another costly experiment? ... Stock Market Sectors Major Markets Heat Map Industry Rankings Industry Heat Map 
Industry Performance Stocks by Grouping. Options. Market Pulse. ... date: 'EEE, MMM dd, yyyy h:mm a' ]] [[ zone ]] 
Reserve ...

[Is Amazon Stock a Buy, Sell, or Hold Ahead of Alexa+ Launch? - 
inkl](https://www.inkl.com/news/is-amazon-stock-a-buy-sell-or-hold-ahead-of-alexa-launch)
Citi pointed to a 20% year-over-year surge in Alexa user engagement across the 600 million Alexa-enabled devices 
sold to date, signaling strong consumer traction. With Alexa+ expected to drive even higher engagement and 
transaction volumes, Citi is keeping Amazon on its top-pick list, reaffirming its "Buy" rating and a $273 price 
target.

[Amazon Announces AI Features For Alexa: Will It Finally Turn ... - 
Benzinga](https://www.benzinga.com/markets/equities/25/02/43990898/amazon-announces-ai-features-for-alexa-will-it-f
inally-turn-around-its-unprofitable-device-business)
What Happened: The Seattle-based company is rolling out an Alexa+ subscription, free for Amazon Prime subscribers, 
within the next few weeks. The subscription costs $19.99 per month for non-Prime ...

[Is Amazon Stock a Buy, Sell, or Hold Ahead of Alexa+ 
Launch?](https://www.theglobeandmail.com/investing/markets/stocks/GOOGL/pressreleases/31215645/is-amazon-stock-a-bu
y-sell-or-hold-ahead-of-alexa-launch/)
Rolling out in early access this month, Alexa+ will cost $19.99 per month, though Amazon Prime members will get it 
for free. This upgraded version isn't just a voice assistant.

[Amazon debuts new Alexa voice assistant with AI 
overhaul](https://www.reuters.com/technology/artificial-intelligence/amazon-eyes-new-direction-alexa-with-ai-overha
ul-2025-02-26/)
Alexa+ is free for Amazon Prime members and is priced at $19.99 a month for non-Prime users. The service will be 
available in March to some users, with the rollout expanding to more people over time.

[Amazon Is Giving Alexa a Major AI Upgrade. Here's What We 
Know](https://www.cnet.com/home/smart-home/amazon-is-giving-alexa-is-a-major-ai-upgrade-heres-what-we-know/)
With more than half a billion Alexa-enabled devices in homes worldwide, a generative AI upgrade could reshape how 
users interact with voice assistants, and potentially introduce a paid tier for ...

[Amazon's AI revamp of Alexa assistant nears unveiling | 
Reuters](https://www.reuters.com/technology/amazon-set-release-long-delayed-alexa-generative-ai-revamp-2025-02-05/)
Amazon prepares to release Alexa generati

[Step 0: Duration 2.92 seconds| Input tokens: 3,433 | Output tokens: 79]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("On the day of the Alexa+ announcement (2025-02-26), Amazon's stock price increased by 0.73%,       
  rising from $212.80 to $214.35. The stock experienced a slight positive movement, consistent with the earlier    
  search result mentioning a 0.5% gain. However, the stock price dropped to $208.74 the following day, suggesting  
  mixed market sentiment about the Alexa+ announcement.")                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: On the day of the Alexa+ announcement (2025-02-26), Amazon's stock price increased by 0.73%, 
rising from $212.80 to $214.35. The stock experienced a slight positive movement, consistent with the earlier 
search result mentioning a 0.5% gain. However, the stock price dropped to $208.74 the following day, suggesting 
mixed market sentiment about the Alexa+ announcement.

[Step 4: Duration 4.96 seconds| Input tokens: 26,771 | Output tokens: 1,278]

"On the day of the Alexa+ announcement (2025-02-26), Amazon's stock price increased by 0.73%, rising from $212.80 to $214.35. The stock experienced a slight positive movement, consistent with the earlier search result mentioning a 0.5% gain. However, the stock price dropped to $208.74 the following day, suggesting mixed market sentiment about the Alexa+ announcement."

In [9]:
stock_analysis_agent.run("How has Amazon's stock price changed since the Fed began lowering rates in 2024")

╭───────────────────────────────────────── New run - stock_analyst_agent ─────────────────────────────────────────╮
│                                                                                                                 │
│ How has Amazon's stock price changed since the Fed began lowering rates in 2024                                 │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0 ────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_result = web_search(query="When did Federal Reserve first cut interest rates in 2024")                    
  print(search_result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Federal Funds Rate History 1990 to 2024 - Forbes 
Advisor](https://www.forbes.com/advisor/investing/fed-funds-rate-history/)
The Federal Reserve has battled a variety of economic troubles over the past 35 years. ... the Fed changed course 
in September 2024 when it cut the federal funds rate by 50 basis ... The Fed cut ...

[Federal Reserve made a 3rd consecutive rate cut today. Here's how it 
...](https://www.cbsnews.com/news/federal-reserve-meeting-rate-cut-interest-rates-december/)
On Dec. 18, the Federal Reserve made its third consecutive cut of 2024, reducing the federal funds rate by 0.25 
percentage points. Yet the Fed also projected a slower pace of cuts in 2025, a move ...

[U.S. federal funds rate 1954-2025 | 
Statista](https://www.statista.com/statistics/187616/effective-rate-of-us-federal-funds-monthly/)
The rate remained unchanged for over a year, before the Federal Reserve initiated its first rate cut in nearly 
three years in September 2024, bringing the rate to 5.13 percent.

[Fed Cuts Interest Rates For First Time In 4 Years: Here's ... - 
Forbes](https://www.forbes.com/sites/dereksaul/2024/09/18/fed-cuts-interest-rates-for-first-time-in-4-years-heres-w
hat-it-means-for-you/)
It's the first cut to the federal funds rate since March 2020, bringing rates down to 4.75% to 5% from the 5.25% to
5.5% range they've sat since last July, the highest rates had been since 2001.

[Federal Reserve cuts interest rates for first time in 4 years | Fox 
...](https://www.foxbusiness.com/economy/federal-reserve-interest-rate-decision-september-2024)
The Fed's first interest rate cut since March 2020 lowers the benchmark federal funds rate to a range of 4.75% to 
5%. Interest rates had been at a range of 5.25% to 5.5% since July 2023, the ...

[Fed cuts rates by a half point at September 2024 
meeting](https://www.cnbc.com/2024/09/18/fed-cuts-rates-september-2024-.html)
The Federal Reserve on Wednesday lowered the federal funds rate to a range between 4.75%-5%.

[Why Is the Federal Reserve Reducing Interest Rates? - CRS 
Reports](https://crsreports.congress.gov/product/pdf/IN/IN12427)
August 2019 to August 2024 Source: Federal Reserve, Bureau of Labor Statistics, Bureau of Economic Analysis. 
Although the Fed initially expected to cut rates in the first half of 2024, inflation in the first three months of 
the year unexpectedly rose to 3.4% on annual basis—and was even higher if food and energy prices were removed.

[Fed Cuts Interest Rates for First Time in Four Years - The New York 
Times](https://www.nytimes.com/2024/09/18/business/economy/fed-reserve-interest-rate-cut.html)
Sept. 18, 2024. The Federal Reserve cut interest rates on Wednesday by half a percentage point, an unusually large 
move and a clear signal that central bankers think they are winning their war ...

[Federal Reserve lowers interest rates by 0.50 percentage points in 
...](https://www.cbsnews.com/news/fed-rate-cut-decision-federal-reserve-interest-rates-september-meeting/)
The Fed said the cut lowers the federal funds rate into a range of 4.75% to 5%, down from its prior range of 5.25% 
to 5.5%, which had been its highest level in 23 years.

[September 2024 Fed Meeting: Fed Cuts Rates by Half Point to Support 
...](https://www.jpmorgan.com/insights/outlook/economic-outlook/fed-meeting-september-2024)
At the September 2024 Federal Open Market Committee (FOMC) meeting, the Federal Reserve (Fed) lowered interest 
rates by 50 basis points, easing monetary policy for the first time in four years due to progress on the Fed's dual
mandate. This lowers the interest rate target to a range of 4.75% to 5%. 1

Out: None

[Step 0: Duration 4.72 seconds| Input tokens: 3,431 | Output tokens: 110]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Prepare the final analysis                                                                                     
  analysis = f"""Amazon Stock Performance Since Fed Rate Cut (September 2024):                                     
  - Initial Price (2024-09-18): ${initial_price:.2f}                                                               
  - Final Price (2025-03-05): ${final_price:.2f}                                                                   
  - Total Percentage Change: +{percent_change:.2f}%                                                                
  - Highest Price: ${highest_price:.2f}                                                                            
  - Lowest Price: ${lowest_price:.2f}                                                                              
                                                                                                                   
  Key Observations:                                                                                                
  1. The stock has shown a positive trend since the Fed's rate cut                                                 
  2. Amazon experienced an overall upward movement of 11.76%                                                       
  3. The stock reached a peak of ${highest_price:.2f}, indicating significant growth potential                     
  """                                                                                                              
                                                                                                                   
  final_answer(analysis)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: Amazon Stock Performance Since Fed Rate Cut (September 2024):
- Initial Price (2024-09-18): $186.42999267578125
- Final Price (2025-03-05): $208.36000061035156
- Total Percentage Change: +11.763132970084161%
- Highest Price: $242.52000427246094
- Lowest Price: $180.25

Key Observations:
1. The stock has shown a positive trend since the Fed's rate cut
2. Amazon experienced an overall upward movement of 11.76%
3. The stock reached a peak of $242.52000427246094, indicating significant growth potential

[Step 3: Duration 5.91 seconds| Input tokens: 19,781 | Output tokens: 843]

"Amazon Stock Performance Since Fed Rate Cut (September 2024):\n- Initial Price (2024-09-18): $186.42999267578125\n- Final Price (2025-03-05): $208.36000061035156\n- Total Percentage Change: +11.763132970084161%\n- Highest Price: $242.52000427246094\n- Lowest Price: $180.25\n\nKey Observations:\n1. The stock has shown a positive trend since the Fed's rate cut\n2. Amazon experienced an overall upward movement of 11.76%\n3. The stock reached a peak of $242.52000427246094, indicating significant growth potential\n"

In [10]:
stock_analysis_agent.run("What is the correlation between Amazon's stock price and the GDP of the United States?")

╭───────────────────────────────────────── New run - stock_analyst_agent ─────────────────────────────────────────╮
│                                                                                                                 │
│ What is the correlation between Amazon's stock price and the GDP of the United States?                          │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0 ────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import math                                                                                                      
                                                                                                                   
  # Get Amazon stock closing prices                                                                                
  stock_data = get_ticker_data(                                                                                    
      tickers=["AMZN"],                                                                                            
      start_date="2010-01-01",                                                                                     
      end_date="2023-12-31",                                                                                       
      metric="Close",                                                                                              
      sampling="monthly"                                                                                           
  )                                                                                                                
                                                                                                                   
  # Get US GDP data                                                                                                
  gdp_data = get_fred_data(                                                                                        
      series="GDP",                                                                                                
      start_date="2010-01-01",                                                                                     
      end_date="2023-12-31",                                                                                       
      sampling="monthly"                                                                                           
  )                                                                                                                
                                                                                                                   
  # Print initial data to verify                                                                                   
  print("Stock data points:", len(stock_data['AMZN']))                                                             
  print("GDP data points:", len(gdp_data))                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

[*********************100%***********************]  1 of 1 completed


Execution logs:
Stock data points: 168
GDP data points: 166

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Prepare the final answer                                                                                       
  interpretation = f"""                                                                                            
  Correlation Analysis between Amazon Stock Price and US GDP:                                                      
  - Correlation Coefficient (Beta): 59.29                                                                          
  - Interpretation: There is a strong positive correlation between Amazon's stock price and US GDP                 
  - This suggests that as the US GDP increases, Amazon's stock price tends to increase proportionally              
  - A beta of 59.29 indicates that for every 1 unit increase in GDP, Amazon's stock price tends to increase by     
  59.29 units                                                                                                      
  """                                                                                                              
                                                                                                                   
  # Use final_answer tool                                                                                          
  final_answer(interpretation)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 
Correlation Analysis between Amazon Stock Price and US GDP:
- Correlation Coefficient (Beta): 59.29
- Interpretation: There is a strong positive correlation between Amazon's stock price and US GDP
- This suggests that as the US GDP increases, Amazon's stock price tends to increase proportionally
- A beta of 59.29 indicates that for every 1 unit increase in GDP, Amazon's stock price tends to increase by 59.29 
units

[Step 4: Duration 4.69 seconds| Input tokens: 22,064 | Output tokens: 1,068]

"\nCorrelation Analysis between Amazon Stock Price and US GDP:\n- Correlation Coefficient (Beta): 59.29\n- Interpretation: There is a strong positive correlation between Amazon's stock price and US GDP\n- This suggests that as the US GDP increases, Amazon's stock price tends to increase proportionally\n- A beta of 59.29 indicates that for every 1 unit increase in GDP, Amazon's stock price tends to increase by 59.29 units\n"

In [11]:
stock_analysis_agent.run("Compare and analyze the market beta for FAANG stocks since 2019")

╭───────────────────────────────────────── New run - stock_analyst_agent ─────────────────────────────────────────╮
│                                                                                                                 │
│ Compare and analyze the market beta for FAANG stocks since 2019                                                 │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/us.anthropic.claude-3-5-haiku-20241022-v1:0 ────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import datetime                                                                                                  
  import statistics                                                                                                
                                                                                                                   
  # FAANG stocks and S&P 500 ticker                                                                                
  tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL', '^GSPC']                                                     
  start_date = '2019-01-01'                                                                                        
  end_date = '2023-12-31'                                                                                          
                                                                                                                   
  # Retrieve stock data                                                                                            
  stock_data = get_ticker_data(tickers, start_date, end_date, metric='Close', sampling='monthly')                  
  print("Data retrieved successfully.")                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

[*********************100%***********************]  6 of 6 completed


Execution logs:
Data retrieved successfully.

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Prepare detailed interpretation                                                                                
  interpretation = f"""FAANG Stocks Beta Analysis (2019-2023):                                                     
                                                                                                                   
  1. Overall Market Sensitivity:                                                                                   
     - Average Beta: {statistics.mean(beta_values):.2f}                                                            
     - All stocks show higher than market volatility (beta > 1)                                                    
                                                                                                                   
  2. Individual Stock Beta Breakdown:                                                                              
     - Apple (AAPL): {stock_betas['AAPL']:.2f} - Highest market sensitivity                                        
     - Netflix (NFLX): {stock_betas['NFLX']:.2f} - Second most volatile                                            
     - Meta (META): {stock_betas['META']:.2f} - Moderate market sensitivity                                        
     - Amazon (AMZN): {stock_betas['AMZN']:.2f} - Moderate market sensitivity                                      
     - Google (GOOGL): {stock_betas['GOOGL']:.2f} - Least volatile among FAANG                                     
                                                                                                                   
  3. Key Insights:                                                                                                 
     - All FAANG stocks are more volatile than the overall market                                                  
     - Beta range: {min(beta_values):.2f} to {max(beta_values):.2f}                                                
     - Apple shows the most significant market-correlated price movements                                          
  """                                                                                                              
                                                                                                                   
  final_answer(interpretation)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: FAANG Stocks Beta Analysis (2019-2023):

1. Overall Market Sensitivity:
   - Average Beta: 1.1765877840767989
   - All stocks show higher than market volatility (beta > 1)

2. Individual Stock Beta Breakdown:
   - Apple (AAPL): 1.3144159348538609 - Highest market sensitivity
   - Netflix (NFLX): 1.2155703090641945 - Second most volatile
   - Meta (META): 1.1521977986272811 - Moderate market sensitivity
   - Amazon (AMZN): 1.142308728694635 - Moderate market sensitivity
   - Google (GOOGL): 1.0584461491440231 - Least volatile among FAANG

3. Key Insights:
   - All FAANG stocks are more volatile than the overall market
   - Beta range: 1.0584461491440231 to 1.3144159348538609
   - Apple shows the most significant market-correlated price movements

[Step 3: Duration 7.96 seconds| Input tokens: 17,479 | Output tokens: 1,187]

'FAANG Stocks Beta Analysis (2019-2023):\n\n1. Overall Market Sensitivity:\n   - Average Beta: 1.1765877840767989\n   - All stocks show higher than market volatility (beta > 1)\n\n2. Individual Stock Beta Breakdown:\n   - Apple (AAPL): 1.3144159348538609 - Highest market sensitivity\n   - Netflix (NFLX): 1.2155703090641945 - Second most volatile\n   - Meta (META): 1.1521977986272811 - Moderate market sensitivity\n   - Amazon (AMZN): 1.142308728694635 - Moderate market sensitivity\n   - Google (GOOGL): 1.0584461491440231 - Least volatile among FAANG\n\n3. Key Insights:\n   - All FAANG stocks are more volatile than the overall market\n   - Beta range: 1.0584461491440231 to 1.3144159348538609\n   - Apple shows the most significant market-correlated price movements\n'